In [1]:
import pandas as pd
import numpy as np
import time
from matplotlib import pyplot as plt
import seaborn as sns
from countryinfo import CountryInfo

In [2]:
train = pd.read_csv("../data/train/train.csv")

In [3]:
train.head()

,Unnamed: 0,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,month,day
0,479,1916-07-02,Chile,Uruguay,0.0,4.0,COPA,Buenos Aires,Argentina,True,1916,7,2
1,481,1916-07-06,Argentina,Chile,6.0,1.0,COPA,Buenos Aires,Argentina,False,1916,7,6
2,482,1916-07-08,Brazil,Chile,1.0,1.0,COPA,Buenos Aires,Argentina,True,1916,7,8
3,483,1916-07-10,Argentina,Brazil,1.0,1.0,COPA,Buenos Aires,Argentina,False,1916,7,10
4,485,1916-07-12,Brazil,Uruguay,1.0,2.0,COPA,Buenos Aires,Argentina,True,1916,7,12


In [4]:
train.drop(columns=['Unnamed: 0'], inplace=True)

In [5]:
train['result'] = np.select(
    [
        train['home_score'] > train['away_score'],
        train['home_score'] < train['away_score']
    ],
    [
        1,  
        2   
    ],
    default=0 
)

In [6]:
train.drop(columns=['home_score', 'away_score'], inplace=True)

## FIFA Rankings

In [7]:
fifa = pd.read_csv('../data/raw/fifa_ranking.csv')
min_date = fifa['rank_date'].min()
train = train[train['date'] >= min_date]

In [19]:

all_teams = pd.concat([train['home_team'], train['away_team']]).unique()

missing_teams = [team for team in all_teams if team not in fifa['country_full'].values]

print(f"Total unique teams in train: {len(all_teams)}")
print(f"Teams NOT found in fifa['country_full']: {len(missing_teams)}")
print(missing_teams)

Total unique teams in train: 147
Teams NOT found in fifa['country_full']: 11
['United States', 'Ivory Coast', 'DR Congo', 'South Korea', 'Czech Republic', 'Iran', 'China', 'North Korea', 'Cape Verde', 'Kyrgyzstan', 'Gambia']


## Location

In [8]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(
    user_agent="nations-matches-prediction",
    timeout=10
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=2,   # Nominatim requests at most ~1/sec
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=False
)

In [9]:
def get_capital(country):
    try:
        return CountryInfo(country).capital()
    except:
        return None

In [10]:
def get_log_lat(city):
    try:
        location = geolocator.geocode(city)
        print(location.latitude, location.longitude)
        return location.latitude, location.longitude
    except Exception as e:
        raise e
        return None

In [11]:
train['home_capital'] = train['home_team'].apply(get_capital)
train['away_capital'] = train['away_team'].apply(get_capital)

In [12]:
train['home_comb'] = train['home_capital'] + ", " + train['home_team']
train['away_comb'] = train['away_capital'] + ", " + train['away_team']
train['stadium_comb'] = train['city'] + ", " + train['country']

In [13]:
train['home_comb']

1608             Quito, Ecuador
1609           Bogotá, Colombia
1610        Montevideo, Uruguay
1611    Buenos Aires, Argentina
1612           Brasília, Brazil
                 ...           
4009            Zagreb, Croatia
4010              Madrid, Spain
4011        Ljubljana, Slovenia
4012        Copenhagen, Denmark
4013    Buenos Aires, Argentina
Name: home_comb, Length: 2406, dtype: str

In [14]:
train['stadium_comb']

1608            Quito, Ecuador
1609          Machala, Ecuador
1610           Ambato, Ecuador
1611        Guayaquil, Ecuador
1612           Cuenca, Ecuador
                 ...          
4009          Hamburg, Germany
4010    Gelsenkirchen, Germany
4011           Munich, Germany
4012        Frankfurt, Germany
4013    Atlanta, United States
Name: stadium_comb, Length: 2406, dtype: str

In [15]:
import time
import json
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderRateLimited

geolocator = Nominatim(
    user_agent="nations-matches-prediction",
    timeout=10
)

CACHE_FILE = "coords_cache.json"

# Load existing cache so you never re-geocode the same place
try:
    with open(CACHE_FILE) as f:
        coords = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    coords = {}

def get_coords(place):
    if place in coords:
        return coords[place]  # skip already-geocoded places
    
    for attempt in range(5):
        try:
            time.sleep(1.2)  # always wait before each request
            location = geolocator.geocode(place)
            result = (location.latitude, location.longitude) if location else None
            coords[place] = result

            # Save after every successful call
            with open(CACHE_FILE, "w") as f:
                json.dump(coords, f)

            return result

        except GeocoderRateLimited:
            wait = 60
            print(f"Rate limited on '{place}', waiting {wait}s (attempt {attempt+1}/5)")
            time.sleep(wait)

    print(f"Giving up on '{place}' after 5 attempts")
    coords[place] = None
    return None

places = pd.concat([
    train['home_comb'],
    train['away_comb'],
    train['stadium_comb']
]).dropna().unique()

for place in places:
    result = get_coords(place)
    print(f"{place}: {result}")

Quito, Ecuador: [-0.2201641, -78.5123274]
Bogotá, Colombia: [4.6533817, -74.0836331]
Montevideo, Uruguay: [-34.9058916, -56.1913095]
Buenos Aires, Argentina: [-34.6095579, -58.3887904]
Brasília, Brazil: [-15.7939869, -47.8828]
Asunción, Paraguay: [-25.2800459, -57.6343814]
Washington D.C., United States: [38.8950982, -77.0363849]
Mexico City, Mexico: [19.3207722, -99.1514678]
Santiago, Chile: [-33.4376995, -70.6510671]
Lima, Peru: [-12.0459808, -77.0305912]
Abuja, Nigeria: [9.0643305, 7.4892974]
Tunis, Tunisia: [36.8002068, 10.1857757]
Accra, Ghana: [5.5571096, -0.2012376]
Yamoussoukro, Ivory Coast: [6.8200066, -5.2776034]
Cairo, Egypt: [30.0443879, 31.2357257]
Bamako, Mali: [12.6132656, -7.9847391]
Dakar, Senegal: [14.693425, -17.447938]
Lusaka, Zambia: [-15.4163395, 28.2818414]
Kinshasa, DR Congo: [-4.3196982, 15.3424196]
Berlin, Germany: [52.5173885, 13.3951309]
Madrid, Spain: [40.416782, -3.703507]
Rome, Italy: [41.8933203, 12.4829321]
Brussels, Belgium: [50.8467372, 4.352493]
Yaou

In [16]:
train[['home_lat', 'home_lon']] = (
    train['home_comb']
    .map(coords)
    .apply(pd.Series)
)

train[['away_lat', 'away_lon']] = (
    train['away_comb']
    .map(coords)
    .apply(pd.Series)
)

train[['stadium_lat', 'stadium_lon']] = (
    train['stadium_comb']
    .map(coords)
    .apply(pd.Series)
)

In [17]:
train.head()

,date,home_team,away_team,tournament,city,country,neutral,year,month,day,...,away_capital,home_comb,away_comb,stadium_comb,home_lat,home_lon,away_lat,away_lon,stadium_lat,stadium_lon
1608,1993-06-15,Ecuador,Venezuela,COPA,Quito,Ecuador,False,1993,6,15,...,Caracas,"Quito, Ecuador","Caracas, Venezuela","Quito, Ecuador",-0.220164,-78.512327,10.506093,-66.914601,-0.220164,-78.512327
1609,1993-06-16,Colombia,Mexico,COPA,Machala,Ecuador,True,1993,6,16,...,Mexico City,"Bogotá, Colombia","Mexico City, Mexico","Machala, Ecuador",4.653382,-74.083633,19.320772,-99.151468,-3.314894,-79.951280
1610,1993-06-16,Uruguay,United States,COPA,Ambato,Ecuador,True,1993,6,16,...,Washington D.C.,"Montevideo, Uruguay","Washington D.C., United States","Ambato, Ecuador",-34.905892,-56.191310,38.895098,-77.036385,-1.242241,-78.628759
1611,1993-06-17,Argentina,Bolivia,COPA,Guayaquil,Ecuador,True,1993,6,17,...,Sucre,"Buenos Aires, Argentina","Sucre, Bolivia","Guayaquil, Ecuador",-34.609558,-58.388790,-19.047725,-65.259431,-2.190057,-79.886867
1612,1993-06-18,Brazil,Peru,COPA,Cuenca,Ecuador,True,1993,6,18,...,Lima,"Brasília, Brazil","Lima, Peru","Cuenca, Ecuador",-15.793987,-47.882800,-12.045981,-77.030591,-2.897407,-79.004173


## Weather

In [18]:
import requests

def get_weather_url(lat, lon, date):
        
    try:
        history = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                    "latitude": lat,
                    "longitude": lon,
                    "start_date": date,
                    "end_date": date,
                    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max",
                    "timezone": "auto"
                }
            ).json()
        return {
            'temperature_max' : history['daily']['temperature_2m_max'][0],
            'temperature_min' : history['daily']['temperature_2m_min'][0],
            'precipitation' : history['daily']['precipitation_sum'][0],
            'wind_speed' : history['daily']['wind_speed_10m_max'][0]
        }
    except:
        raise Exception("Error")

In [ ]:
import json

CACHE_FILE = "weather_cache.json"

# Load cache
try:
    with open(CACHE_FILE) as f:
        weather = json.load(f)
except FileNotFoundError:
    weather = {}

def cache_key(lat, lon, date):
    return f"{lat}_{lon}_{date}"  

def get_weather(lat, lon, date):
    key = cache_key(lat, lon, date)

    if key in weather:
        print(f"Cache hit for {key}")
        return weather[key]  

    for attempt in range(5):
        try:
            time.sleep(1.2)  # respect 600/min limit (~1 req/sec)
            result = get_weather_url(lat, lon, date)

            weather[key] = result
            print("result is back for ", (lon, lat, date))
            with open(CACHE_FILE, "w") as f:
                json.dump(weather, f)

            return result

        except Exception as e:
            wait = 60
            print(f"Error: {e}, waiting {wait}s (attempt {attempt+1}/5)")
            time.sleep(wait)

    print(f"Giving up on ({lat}, {lon}, {date}) after 5 attempts")
    weather[key] = None
    return None

In [20]:
train.dropna(inplace=True)

In [ ]:

unique = train[['home_lat', 'home_lon', 'date']].drop_duplicates()

unique['weather'] = unique.apply(
    lambda row: get_weather(row['home_lat'], row['home_lon'], row['date']),
    axis=1
)

weather_df = unique['weather'].apply(pd.Series)
unique = pd.concat([unique, weather_df], axis=1).drop(columns='weather')

train = train.merge(unique, on=['home_lat', 'home_lon', 'date'], how='left')

Cache hit for -0.2201641_-78.5123274_1993-06-15
Cache hit for 4.6533817_-74.0836331_1993-06-16
Cache hit for -34.9058916_-56.1913095_1993-06-16
Cache hit for -34.6095579_-58.3887904_1993-06-17
Cache hit for -15.7939869_-47.8828_1993-06-18
Cache hit for -25.2800459_-57.6343814_1993-06-18
Cache hit for -0.2201641_-78.5123274_1993-06-19
Cache hit for -34.9058916_-56.1913095_1993-06-19
Cache hit for -34.6095579_-58.3887904_1993-06-20
Cache hit for 4.6533817_-74.0836331_1993-06-20
Cache hit for -15.7939869_-47.8828_1993-06-21
Cache hit for -25.2800459_-57.6343814_1993-06-21
Cache hit for -0.2201641_-78.5123274_1993-06-22
Cache hit for 38.8950982_-77.0363849_1993-06-22
Cache hit for -34.6095579_-58.3887904_1993-06-23
Cache hit for 19.3207722_-99.1514678_1993-06-23
Cache hit for -15.7939869_-47.8828_1993-06-24
Cache hit for -33.4376995_-70.6510671_1993-06-24
Cache hit for 4.6533817_-74.0836331_1993-06-26
Cache hit for -0.2201641_-78.5123274_1993-06-26
Cache hit for -15.7939869_-47.8828_1993-0

In [ ]:

unique = train[['away_lat', 'away_lon', 'date']].drop_duplicates()

unique['weather'] = unique.apply(
    lambda row: get_weather(row['away_lat'], row['away_lon'], row['date']),
    axis=1
)

weather_df = unique['weather'].apply(pd.Series)
unique = pd.concat([unique, weather_df], axis=1).drop(columns='weather')

train = train.merge(unique, on=['away_lat', 'away_lon', 'date'], how='left')

In [ ]:

unique = train[['stadium_lat', 'stadium_lon', 'date']].drop_duplicates()

unique['weather'] = unique.apply(
    lambda row: get_weather(row['stadium_lat'], row['stadium_lon'], row['date']),
    axis=1
)

weather_df = unique['weather'].apply(pd.Series)
unique = pd.concat([unique, weather_df], axis=1).drop(columns='weather')

train = train.merge(unique, on=['stadium_lat', 'stadium_lon', 'date'], how='left')